# Geospatial POI and Routing Features

## Objective

This notebook develops external geographic features for Victorian
rental properties, including:

- distance to nearest train station
- distance to Melbourne CBD
- proximity to schools
- proximity to parks
- proximity to shopping/amenities

Straight-line distance is used as an initial baseline.
OpenRouteService route distance will subsequently be calculated
at suburb level to reduce API usage.

In [2]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.neighbors import BallTree

ModuleNotFoundError: No module named 'sklearn'

## 1. File paths

Define the locations of the raw and curated datasets.

In [ ]:
PROJECT_ROOT = Path("..")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CURATED_DIR = PROJECT_ROOT / "data" / "curated"

TRANSPORT_DIR = RAW_DIR / "transport"
SCHOOL_DIR = RAW_DIR / "schools"
OSM_DIR = RAW_DIR / "osm"

CURATED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Victorian train stations

Transport Victoria GTFS data are divided by transport mode.

The metropolitan train dataset and regional train dataset are combined
so that train accessibility can be calculated for properties throughout Victoria.

Only station-level locations are retained where possible, rather than
individual platforms or entrances.

In [ ]:
metro_train_path = (
    TRANSPORT_DIR
    / "gtfs"
    / "1"
    / "google_transit"
    / "stops.txt"
)

regional_train_path = (
    TRANSPORT_DIR
    / "gtfs"
    / "2"
    / "google_transit"
    / "stops.txt"
)

metro_stops = pd.read_csv(metro_train_path)
regional_stops = pd.read_csv(regional_train_path)

print("Metro stop records:", len(metro_stops))
print("Regional stop records:", len(regional_stops))

Metro stop records: 627
Regional stop records: 2859


In [ ]:
train_stops = pd.concat(
    [metro_stops, regional_stops],
    ignore_index=True
)

train_stops.head()

,stop_id,stop_name,stop_lat,stop_lon,stop_url,location_type,parent_station,wheelchair_boarding,level_id,platform_code
0,11212,Flinders Street Station,-37.818095,144.966266,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
1,11213,Flinders Street Station,-37.818144,144.966492,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
2,11214,Flinders Street Station,-37.818198,144.966524,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
3,11215,Flinders Street Station,-37.818289,144.966556,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN
4,11216,Flinders Street Station,-37.818326,144.966600,https://transport.vic.gov.au/stop/1071/?utm_so...,NaN,vic:rail:FSS,1.0,Level 0,NaN


In [ ]:
if "location_type" in train_stops.columns:
    stations = train_stops[
        train_stops["location_type"] == 1
    ].copy()
else:
    stations = train_stops.copy()


stations = stations[
    [
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ]
].copy()

stations = stations.rename(columns={
    "stop_lat": "latitude",
    "stop_lon": "longitude"
})


stations = stations.dropna(
    subset=["latitude", "longitude"]
)

stations = stations.drop_duplicates(
    subset=[
        "stop_name",
        "latitude",
        "longitude"
    ]
).reset_index(drop=True)

print("Number of train stations:", len(stations))

stations.head(10)


Number of train stations: 320


,stop_id,stop_name,latitude,longitude
0,nsw:rail:ABY,Albury Railway Station,-36.085046,146.924381
1,vic:rail:ARR,Ardeer Railway Station,-37.783065,144.802188
2,vic:rail:ART,Ararat Railway Station,-37.282241,142.936912
3,vic:rail:AVL,Avenel Railway Station,-36.893648,145.229515
4,vic:rail:BAH,Bacchus Marsh Railway Station,-37.687578,144.436785
5,vic:rail:BAT-V,Ballarat Railway Station,-37.558791,143.859457
6,vic:rail:BDE,Bairnsdale Railway Station,-37.828720,147.627614
7,vic:rail:BET,Beaufort Railway Station,-37.427627,143.382381
8,vic:rail:BEW,Berwick Railway Station,-38.039980,145.345417
9,vic:rail:BGE,Birregurra Railway Station,-38.328808,143.783625


## 3. Victorian school locations

The Victorian School Locations dataset contains primary and secondary
school locations throughout Victoria.

School coordinates are used to calculate:

- distance to the nearest school
- number of schools within 2 km

These variables are used as measures of access to education and local
liveability.

In [ ]:
school_path = (
    SCHOOL_DIR
    / "dv402-SchoolLocations2025.csv"
)

schools = pd.read_csv(school_path)

print("Number of school records:", len(schools))

schools.head()

Number of school records: 2301


,Education_Sector,Entity_Type,School_No,School_Name,School_Type,School_Status,Address_Line_1,Address_Line_2,Address_Town,Address_State,...,Postal_State,Postal_Postcode,Full_Phone_No,Region,Area,LGA_ID,LGA_Name,LGA_TYPE,X,Y
0,Catholic,2,20,Parade College,Secondary,O,1436 Plenty Road,NaN,BUNDOORA,VIC,...,VIC,3083,03 9468 3300,NORTH-WESTERN VICTORIA,North Eastern Melbourne,66,Banyule (C),Metro,145.066978,-37.690178
1,Catholic,2,25,Simonds Catholic College,Secondary,O,273 Victoria Street,NaN,WEST MELBOURNE,VIC,...,VIC,3003,03 9321 9200,SOUTH-WESTERN VICTORIA,Western Melbourne,460,Melbourne (C),Metro,144.952883,-37.805971
2,Catholic,2,26,St Mary’s College Melbourne,Secondary,O,11 Westbury Street,NaN,ST KILDA EAST,VIC,...,VIC,3182,03 9529 6611,SOUTH-EASTERN VICTORIA,Bayside Peninsula,590,Port Phillip (C),Metro,144.997001,-37.859365
3,Catholic,2,28,St Patrick's College Ballarat,Secondary,O,1431 Sturt Street,NaN,BALLARAT,VIC,...,VIC,3350,03 5331 1688,SOUTH-WESTERN VICTORIA,Central Highlands,57,Ballarat (C),Non Metro,143.831558,-37.559711
4,Catholic,2,29,St Patrick's School,Primary,O,119 Drummond Street South,NaN,BALLARAT,VIC,...,VIC,3350,03 5332 7680,SOUTH-WESTERN VICTORIA,Central Highlands,57,Ballarat (C),Non Metro,143.847147,-37.564397


In [ ]:
schools = schools[
    [
        "School_No",
        "School_Name",
        "School_Type",
        "Education_Sector",
        "Address_Town",
        "Address_Postcode",
        "X",
        "Y"
    ]
].copy()

schools = schools.rename(columns={
    "X": "longitude",
    "Y": "latitude"
})

schools = schools.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)


## 4. Temporary locations for pipeline development

A small set of Victorian suburbs is used to test the geographic feature
pipeline before the group's complete rental property dataset is available.

These coordinates are temporary development values and will later be
replaced with property coordinates or suburb centroids.

In [ ]:
test_locations = pd.DataFrame({
    "suburb": [
        "Carlton",
        "Richmond",
        "Footscray",
        "Box Hill",
        "Werribee"
    ],

    "latitude": [
        -37.800,
        -37.818,
        -37.802,
        -37.819,
        -37.900
    ],

    "longitude": [
        144.967,
        145.002,
        144.900,
        145.122,
        144.661
    ]
})

test_locations

,suburb,latitude,longitude
0,Carlton,-37.800,144.967
1,Richmond,-37.818,145.002
2,Footscray,-37.802,144.900
3,Box Hill,-37.819,145.122
4,Werribee,-37.900,144.661


## 5. Straight-line distance to points of interest

Nearest points of interest are identified using a BallTree with the
Haversine distance metric.

This provides an efficient measure of straight-line geographic distance
between suburb/property locations and external amenities.

In [ ]:
EARTH_RADIUS_KM = 6371.0088

def add_nearest_poi(
    locations,
    pois,
    poi_name=None,
    prefix="poi"
):

    locations = locations.copy()
    pois = pois.copy()

    location_coords = np.radians(
        locations[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    poi_coords = np.radians(
        pois[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    tree = BallTree(
        poi_coords,
        metric="haversine"
    )

    distances, indices = tree.query(
        location_coords,
        k=1
    )

    locations[f"{prefix}_distance_km"] = (
        distances[:, 0]
        * EARTH_RADIUS_KM
    )

    if poi_name is not None:
        locations[f"nearest_{prefix}"] = (
            pois
            .iloc[indices[:, 0]][poi_name]
            .to_numpy()
        )

    return locations

## 6. Distance to nearest train station

For each location, the closest metropolitan or regional train station
is identified.

Straight-line distance is used initially as the baseline accessibility
measure.

In [ ]:
test_locations = add_nearest_poi(
    locations=test_locations,
    pois=stations,
    poi_name="stop_name",
    prefix="train"
)

test_locations[
    [
        "suburb",
        "nearest_train",
        "train_distance_km"
    ]
]

,suburb,nearest_train,train_distance_km
0,Carlton,Parkville Railway Station,0.655380
1,Richmond,West Richmond Railway Station,0.989123
2,Footscray,Footscray Railway Station,0.189086
3,Box Hill,Box Hill Railway Station,0.055890
4,Werribee,Werribee Railway Station,0.069929


## 7. Distance to nearest school

The closest Victorian school is identified for each location.

This feature measures access to nearby educational facilities.

In [ ]:
test_locations = add_nearest_poi(
    locations=test_locations,
    pois=schools,
    poi_name="School_Name",
    prefix="school"
)

test_locations[
    [
        "suburb",
        "nearest_school",
        "school_distance_km"
    ]
]

,suburb,nearest_school,school_distance_km
0,Carlton,Carlton Gardens Primary School,0.317048
1,Richmond,Richmond High School,0.096545
2,Footscray,Footscray City Primary School,0.489763
3,Box Hill,Our Lady of Sion College,0.699279
4,Werribee,Wyndham Community and Education Centre Inc | J...,0.247769


## 8. Number of nearby points of interest

In addition to nearest-distance measures, amenities within a specified
radius are counted.

For schools, a 2 km radius is used initially as a measure of local
educational accessibility.

In [ ]:
def count_pois_within_radius(
    locations,
    pois,
    radius_km,
    prefix
):

    locations = locations.copy()

    location_coords = np.radians(
        locations[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    poi_coords = np.radians(
        pois[
            ["latitude", "longitude"]
        ].to_numpy()
    )

    tree = BallTree(
        poi_coords,
        metric="haversine"
    )

    radius_radians = (
        radius_km
        / EARTH_RADIUS_KM
    )

    neighbours = tree.query_radius(
        location_coords,
        r=radius_radians
    )

    locations[
        f"{prefix}_within_{radius_km:g}km"
    ] = [
        len(points)
        for points in neighbours
    ]

    return locations

In [ ]:
test_locations = count_pois_within_radius(
    locations=test_locations,
    pois=schools,
    radius_km=2,
    prefix="schools"
)

test_locations[
    [
        "suburb",
        "nearest_school",
        "school_distance_km",
        "schools_within_2km"
    ]
]

,suburb,nearest_school,school_distance_km,schools_within_2km
0,Carlton,Carlton Gardens Primary School,0.317048,18
1,Richmond,Richmond High School,0.096545,14
2,Footscray,Footscray City Primary School,0.489763,7
3,Box Hill,Our Lady of Sion College,0.699279,13
4,Werribee,Wyndham Community and Education Centre Inc | J...,0.247769,8


## 9. Distance to Melbourne CBD

Distance to Melbourne CBD is used as a measure of accessibility to the
central employment and commercial area.

Straight-line distance is calculated first. Route-based distance will
later be calculated using OpenRouteService.

In [ ]:
CBD = pd.DataFrame({
    "name": [
        "Melbourne CBD"
    ],

    "latitude": [
        -37.8136
    ],

    "longitude": [
        144.9631
    ]
})

test_locations = add_nearest_poi(
    locations=test_locations,
    pois=CBD,
    poi_name="name",
    prefix="cbd"
)

test_locations[
    [
        "suburb",
        "cbd_distance_km"
    ]
]

,suburb,cbd_distance_km
0,Carlton,1.550582
1,Richmond,3.451924
2,Footscray,5.691551
3,Box Hill,13.970995
4,Werribee,28.208879


## 10. Initial geographic feature table

The following table combines the initial geographic accessibility
features created from the external datasets.

In [ ]:
test_locations[
    [
        "suburb",

        "nearest_train",
        "train_distance_km",

        "nearest_school",
        "school_distance_km",
        "schools_within_2km",

        "cbd_distance_km"
    ]
]

,suburb,nearest_train,train_distance_km,nearest_school,school_distance_km,schools_within_2km,cbd_distance_km
0,Carlton,Parkville Railway Station,0.655380,Carlton Gardens Primary School,0.317048,18,1.550582
1,Richmond,West Richmond Railway Station,0.989123,Richmond High School,0.096545,14,3.451924
2,Footscray,Footscray Railway Station,0.189086,Footscray City Primary School,0.489763,7,5.691551
3,Box Hill,Box Hill Railway Station,0.055890,Our Lady of Sion College,0.699279,13,13.970995
4,Werribee,Werribee Railway Station,0.069929,Wyndham Community and Education Centre Inc | J...,0.247769,8,28.208879
